In [31]:

import os
import certifi
import requests

In [32]:

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
#---- errors ----
# pip install --upgrade pip setuptools wheel
# pip install --upgrade langchain
# pip install --only-binary=:all: cryptography

In [33]:
from langchain.agents import create_react_agent,AgentExecutor


In [34]:

load_dotenv(override=True)
openai_api_key = os.getenv("OPEN_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY").strip()
hf_token = os.getenv("HF_TOKEN").strip()
gemini_api_key = os.getenv("GEMINI_API_KEY", "").strip()
WEATHER_API_KEY=os.getenv("WEATHER_API_KEY", "").strip()
os.environ["SSL_CERT_FILE"] = certifi.where()

In [35]:
search_tool = TavilySearchResults(max_results=3, tavily_api_key=tavily_api_key)


In [36]:
import requests

response = requests.post(
    "https://api.tavily.com/search",
    json={
        "api_key": tavily_api_key,
        "query": "test",
        "max_results": 1
    },
    timeout=30
)

print("HTTP status:", response.status_code)

try:
    print("Response:", response.json())
except ValueError:
    print("Response was not JSON.")

HTTP status: 200
Response: {'query': 'test', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://en.wikipedia.org/wiki/Test', 'title': 'Test - Wikipedia', 'content': '## Sports\n\n Test cricket, a series of matches played by two national representative teams\n Test match (rugby league) "Test match (rugby league)"), a match between teams of the Rugby League International Federation\n Test match (rugby union) "Test match (rugby union)"), an international match usually played between two senior national teams\n Test (greyhound competition) "Test (greyhound competition)"), a greyhound race run between 1941 and 2008\n\n## Other uses [...] Wiktionary logo\n\nLook up test, testing, Test, or TEST in Wiktionary, the free dictionary.\n\nTest(s), testing, or TEST most often refers to:\n\n Test (assessment) "Test (assessment)"), an educational assessment intended to measure the respondents\' knowledge or other abilities\n\nTest(s), testing, or TEST may also refe

In [ ]:
result=search_tool.invoke("give me a list of the top 10 AI news")
result

In [ ]:
llm=chat=ChatOpenAI(model_name="gpt-4",
                    temperature=0,
                    openai_api_key=openai_api_key)

In [ ]:
# HUGGING FACE CONNECTION ------------------------
#------------------------------------------------

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=hf_token,
    base_url="https://router.huggingface.co/v1",
    temperature=0.2
)
result=llm.invoke("What is the capital of France?")
result

In [37]:
#---------google connection --------------

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=gemini_api_key,
    temperature=0
)

In [ ]:
#----- GOOGLE LLM CONNECTION -----
#--------------------------------------
from google import genai


client = genai.Client(api_key=gemini_api_key)
response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="who is Anitha Gandluri"
    )

print(response.text)

In [38]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to these tools:

{tools}

Use this format:

Question: the input question
Thought: your reasoning about what to do
Action: the tool to use, must be one of [{tool_names}]
Action Input: the input to the tool
Observation: the tool result
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the answer to the original question

Question: {input}
Thought:{agent_scratchpad}""")
prompt


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], template='Answer the following questions as best you can. You have access to these tools:\n\n{tools}\n\nUse this format:\n\nQuestion: the input question\nThought: your reasoning about what to do\nAction: the tool to use, must be one of [{tool_names}]\nAction Input: the input to the tool\nObservation: the tool result\n... (repeat Thought/Action/Action Input/Observation as needed)\nThought: I now know the final answer\nFinal Answer: the answer to the original question\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:
from langchain_core.tools import tool
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
       
        f"https://api.weatherapi.com/v1/current.json?"
        f"key={WEATHER_API_KEY}&q={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"temp_c: {data['current']['temp_c']}°C\n"
        f"condition: {data['current']['condition'][0]}\n"
        f"humidity: {data['current']['humidity']}%"
    )
tools = [search_tool, get_weather_data]


In [45]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)



In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
result = agent_executor.invoke({
    "input": "What is the capital of India?"
    "and then find its current weather."
})

print(result["output"])



> Entering new AgentExecutor chain...


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash
Please retry in 19h10m31.369948158s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.8-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, r

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash
Please retry in 19h5m25.334819284s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.8-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 68725
}
]